# Shopify API Authentication

Index:
- [API Keys and Auth](#API-Keys-and-Auth)
- [Demo and Setup](##initial-setup-using-first-and-last-filters)
- [Bulk Operations](## Bulk Operations)

<!-- Initial Setup Using First and Last Filters -->


### API Keys and Auth

In [1]:
# pip install shopifyapp
# !pip install dotenv
# !pip install --upgrade ShopifyAPI
# !pip install --upgrade google-cloud-bigquery

In [2]:
# Import Libraries
import os
import pandas as pd
import requests
import json
import src.utils as utils
import src.queries as queries

# Datetime Packages
from zoneinfo import ZoneInfo
from datetime import datetime as dt
import dateutil.parser as du
import time

# Services Libraries
from shopify_app import ShopifyApp
from google.cloud import bigquery

# Load Shopify Secrets
SHOPIFY_CLIENT_ID = os.getenv("SHOPIFY_CLIENT_ID")
SHOPIFY_SECRET = os.getenv("SHOPIFY_SECRET")
merchant = os.getenv("MERCHANT")
# version = '2022-04'

# Load Google Cloud services account and BigQuery Client
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'cloud_python_private_key.json'

client = bigquery.Client()

## Connect to Shopify

#### Connect to Shopify to get an access token

In [3]:
SHOPIFY_ACCESS_TOKEN = utils.get_credentials(SHOPIFY_CLIENT_ID,SHOPIFY_SECRET)

## Bulk Operations



#### Bulk Operation Query

In [13]:
bulk_query_response = utils.bulk_query_request(queries.test,SHOPIFY_ACCESS_TOKEN)

In [14]:
bulk_query_response

{'data': {'bulkOperationRunQuery': {'bulkOperation': {'id': 'gid://shopify/BulkOperation/7533519536496',
    'status': 'CREATED'},
   'userErrors': []}},
 'extensions': {'cost': {'requestedQueryCost': 10,
   'actualQueryCost': 10,
   'throttleStatus': {'maximumAvailable': 20000.0,
    'currentlyAvailable': 19990,
    'restoreRate': 1000.0}}}}

#### Getting Bulk Operation Status. 

Eventually, will create a function to query to check status and if complete and without errors, then grab the URL with the data.

In [8]:
bulk_status_response = utils.status_update(queries.status,SHOPIFY_ACCESS_TOKEN)

In [9]:
bulk_status_response

{'data': {'currentBulkOperation': {'id': 'gid://shopify/BulkOperation/7533500957040',
   'status': 'COMPLETED',
   'errorCode': None,
   'createdAt': '2026-04-06T23:02:54Z',
   'completedAt': '2026-04-06T23:05:04Z',
   'objectCount': '6337',
   'fileSize': '8775176',
   'url': 'https://storage.googleapis.com/shopify-tiers-assets-prod-us-east1/bulk-operation-outputs/16d4dk4xra1mbtt3hzvecwlvhgt7-final?GoogleAccessId=assets-us-prod%40shopify-tiers.iam.gserviceaccount.com&Expires=1776121504&Signature=nT67YLCq670nxp%2FJNLmhxuvbfUpmaDUo555YkSRxl2qk01jfBzKbHsdOeGgknYiETiVwGfJLy6W0DpshZsNo06vWOZ0o7wkH2wLt2rogZ0z%2F%2Fy0bWoOtQvz1dC%2FLyQ3L1jfL084aUf%2FGgvFw4oCTWxDQkftDC6w85XKb6ZgVq37a7qPa%2FQxEEXkVqAAmLp9n1K0yOG1o3jUoZ2Cp%2BgDR7%2BHgJ%2F0H0bvrpEzf0qwmwmMu00ubgaJD%2FnY17WIQaWhnWdpTmjQYN78WizwDzmIvJC8J%2BBiaUdComID0t7DF6kWbtPeUGGQ5xqVsaguDV60%2BqqUOxN2UAoAXKFAwuSNDow%3D%3D&response-content-disposition=attachment%3B+filename%3D%22bulk-7533500957040.jsonl%22%3B+filename%2A%3DUTF-8%27%27bulk-7533500

In [1]:
def get_link_data(response):

    if (bulk_status_response['data']['currentBulkOperation']['status'] == 'COMPLETED' and bulk_status_response['data']['currentBulkOperation']['errorCode'] is None):
        print("Done! File available at:", bulk_status_response['data']['currentBulkOperation']['url'])
        url_results = bulk_status_response['data']['currentBulkOperation']['url']

    # Source - https://stackoverflow.com/a/22776
    # Posted by PabloG, modified by community. See post 'Timeline' for change history
    # Retrieved 2026-04-02, License - CC BY-SA 4.0

# import urllib.request
    result = requests.get(url_results)
    contents = result.content
    my_json = contents.decode('utf8')
    orders = [json.loads(line) for line in my_json.strip().split('\n') if line.strip()]
    return orders

In [11]:
url_results = get_link_data(bulk_status_response)

Done! File available at: https://storage.googleapis.com/shopify-tiers-assets-prod-us-east1/bulk-operation-outputs/jjmbp1nwzwdovnpegz850bbmz2ng-final?GoogleAccessId=assets-us-prod%40shopify-tiers.iam.gserviceaccount.com&Expires=1776120533&Signature=YZ21vOUcg%2FvHyTxDtEFwAiycNPbYxCXJl5Y8O6Xy9%2BuWAoCznExEdNt%2Fa2ZHg9XdC1u34Li0TIicX7Ha7UKGASd9U9opp9aIrZlLGDFxQKo1Qk7LMVYxQ8gZJYUikj7zEnfJw47V2l8eK1BQ1tGJFfQs35XKULyIMSca0qrcNtWPtk3QwnAKnoE1C9QYUk0xzS7jUgWX0z8WbFHt%2BVrEz46bARRaAfaQUUKSFWaQ8IcNqDaV%2FBwenRAgOt98r3hELGTwCr74txmFTzxd7uv4mTZ5Vnng6TzKKxxggz6%2BKXQKg9MYGTggkN7coZ9AxdCHq56J9eX48J7eVeqnO02Yfw%3D%3D&response-content-disposition=attachment%3B+filename%3D%22bulk-7533463503216.jsonl%22%3B+filename%2A%3DUTF-8%27%27bulk-7533463503216.jsonl&response-content-type=application%2Fjsonl


In [10]:
def poll_for_result(interval_seconds=60, max_attempts=10):
    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}...")
        
        bulk_status_response = utils.status_update(queries.status,SHOPIFY_ACCESS_TOKEN)
        
        if (bulk_status_response['data']['currentBulkOperation']['status'] == 'COMPLETED' and bulk_status_response['data']['currentBulkOperation']['errorCode'] is None):
            print("Done! File available at:", bulk_status_response['data']['currentBulkOperation']['url'])

            url_results = bulk_status_response['data']['currentBulkOperation']['url']
            result = requests.get(url_results)
            contents = result.content
            my_json = contents.decode('utf8')
            orders = [json.loads(line) for line in my_json.strip().split('\n') if line.strip()]
            return orders

        print(f"Status: {bulk_status_response['data']['currentBulkOperation']['status']}. Retrying in {interval_seconds}s...")
        time.sleep(interval_seconds)

    raise TimeoutError("Job did not complete within the allowed attempts.")

In [15]:
orders = poll_for_result()

Attempt 1/10...
Status: RUNNING. Retrying in 60s...
Attempt 2/10...
Status: RUNNING. Retrying in 60s...
Attempt 3/10...
Done! File available at: https://storage.googleapis.com/shopify-tiers-assets-prod-us-east1/bulk-operation-outputs/wgmhlw36uhkdkyee61f7kaidkvd8-final?GoogleAccessId=assets-us-prod%40shopify-tiers.iam.gserviceaccount.com&Expires=1776121768&Signature=a36nav%2Fv7Wb87rwk7jA9j3XgzB4KIeSm7GDhzC0U57wpWUMhIZeshtHMlEAsscD1QXPJ84GkLV7Uk4MuLt%2Bd0zSMG7562Z2stUJ2l0Qdr%2B2hcb6nIzChsa2lajeyGaR6yaggfPhSYSgH5tCNUkl1O2ZGH4Y0%2BrILnW7dW6VNI2glW29QWMykpEgNFpNIue4aLYb1GqsyF%2B65xjGEcEYaOMEPA4AKmm4IUNGDWFOHqL%2BGMW%2BIrtXqhQn%2BnMZcIOJ7ijW1atUUWYaFi4%2F4aEmS82bBKBZQ0UvHCE%2FTiGlF6ftcGEXZoejEE1Fd0H1A4BRCCMSqyUyh2rWmyub%2BLJib5w%3D%3D&response-content-disposition=attachment%3B+filename%3D%22bulk-7533519536496.jsonl%22%3B+filename%2A%3DUTF-8%27%27bulk-7533519536496.jsonl&response-content-type=application%2Fjsonl


In [3]:
# Eventual dataframe
bulk_data = []

# Read jsonl object
with open('test_files/responses/bulk-7515585085808.jsonl') as f:
    for line in f:
        bulk_data.append(json.loads(line))

# Create dataframe from list
bulk_df = pd.DataFrame(bulk_data)

In [9]:
# Source - https://stackoverflow.com/a/22776
# Posted by PabloG, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-02, License - CC BY-SA 4.0

# import urllib.request
result = requests.get(url_results)
contents = result.content
my_json = contents.decode('utf8')
orders = [json.loads(line) for line in my_json.strip().split('\n') if line.strip()]


In [13]:
# orders[0]

In [55]:
test_bulk_data = []

# Read jsonl object
for line in orders:
    test_bulk_data.append(line)

In [13]:


# Resetting data frame
orders_df_complete = pd.DataFrame(columns=['order_id','order_name','email','created_at','cancellation','cancelled_at','cancel_reason','cart_discount_amount_set','channel_information','closed','closed_at','current_cart_discount_amount_set','current_shipping_price_set','current_subtotal_line_items_quantity','current_subtotal_price_set','current_tax_lines','current_total_additional_fees_set','current_total_discounts_set','current_total_price_set','current_total_tax_set','discount_code','discount_codes','display_financial_status','display_fulfillment_status','fully_paid','original_total_price_set','payment_gateway_names','refunds','registered_source_url','return_status','source_name','subtotal_line_items_quantity','subtotal_price_set','total_discounts_set','total_price_set','transactions'
])

# Resetting objects
order_id = []
order_name = []
email = []
created_at = []
cancellation = []
cancelled_at = []
cancel_reason = []
cart_discount_amount_set = []
channel_information = []
closed = []
closed_at = []
current_cart_discount_amount_set = []
current_shipping_price_set = []
current_subtotal_line_items_quantity = []
current_subtotal_price_set = []
current_tax_lines = []
current_total_additional_fees_set = []
current_total_discounts_set = []
current_total_price_set = []
current_total_tax_set = []
discount_code = []
discount_codes = []
display_financial_status = []
display_fulfillment_status = []
fully_paid = []
original_total_price_set = []
payment_gateway_names = []
refunds = []
registered_source_url = []
return_status = []
source_name = []
subtotal_line_items_quantity = []
subtotal_price_set = []
total_discounts_set = []
total_price_set = []
transactions = []

# Looping through data
for response in orders:
    
    order_id.append(response['id'])
    order_name.append(response['name'])
    email.append(response['email'])
    created_at.append(response['createdAt'])
    cancellation.append(response['cancellation'])
    cancelled_at.append(response['cancelledAt'])
    cancel_reason.append(response['cancelReason'])
    cart_discount_amount_set.append(response['cartDiscountAmountSet'])
    channel_information.append(response['channelInformation'])
    closed.append(response['closed'])
    closed_at.append(response['closedAt'])
    current_cart_discount_amount_set.append(response['currentCartDiscountAmountSet'])
    current_shipping_price_set.append(response['currentShippingPriceSet'])
    current_subtotal_line_items_quantity.append(response['currentSubtotalLineItemsQuantity'])
    current_subtotal_price_set.append(response['currentSubtotalPriceSet'])
    current_tax_lines.append(response['currentTaxLines'])
    current_total_additional_fees_set.append(response['currentTotalAdditionalFeesSet'])
    current_total_discounts_set.append(response['currentTotalDiscountsSet'])
    current_total_price_set.append(response['currentTotalPriceSet'])
    current_total_tax_set.append(response['currentTotalTaxSet'])
    discount_code.append(response['discountCode'])
    discount_codes.append(response['discountCodes'])
    display_financial_status.append(response['displayFinancialStatus'])
    display_fulfillment_status.append(response['displayFulfillmentStatus'])
    fully_paid.append(response['fullyPaid'])
    original_total_price_set.append(response['originalTotalPriceSet'])
    payment_gateway_names.append(response['paymentGatewayNames'])
    refunds.append(response['refunds'])
    registered_source_url.append(response['registeredSourceUrl'])
    return_status.append(response['returnStatus'])
    source_name.append(response['sourceName'])
    subtotal_line_items_quantity.append(response['subtotalLineItemsQuantity'])
    subtotal_price_set.append(response['subtotalPriceSet'])
    total_discounts_set.append(response['totalDiscountsSet'])
    total_price_set.append(response['totalPriceSet'])
    transactions.append(response['transactions'])
      
# Populate new table
orders_df_complete['order_id'] = order_id
orders_df_complete['order_name'] = order_name
orders_df_complete['email'] = email
orders_df_complete['created_at'] = created_at
orders_df_complete['cancellation'] = cancellation
orders_df_complete['cancelled_at'] = cancelled_at
orders_df_complete['cancel_reason'] = cancel_reason
orders_df_complete['cart_discount_amount_set'] = cart_discount_amount_set
orders_df_complete['channel_information'] = channel_information
orders_df_complete['closed'] = closed
orders_df_complete['closed_at'] = closed_at
orders_df_complete['current_cart_discount_amount_set'] = current_cart_discount_amount_set
orders_df_complete['current_shipping_price_set'] = current_shipping_price_set    
orders_df_complete['current_subtotal_line_items_quantity'] = current_subtotal_line_items_quantity
orders_df_complete['current_subtotal_price_set'] = current_subtotal_price_set
orders_df_complete['current_tax_lines'] = current_tax_lines
orders_df_complete['current_total_additional_fees_set'] = current_total_additional_fees_set
orders_df_complete['current_total_discounts_set'] = current_total_discounts_set    
orders_df_complete['current_total_price_set'] = current_total_price_set
orders_df_complete['current_total_tax_set'] = current_total_tax_set
orders_df_complete['discount_code'] = discount_code
orders_df_complete['discount_codes'] = discount_codes
orders_df_complete['display_financial_status'] = display_financial_status    
orders_df_complete['display_fulfillment_status'] = display_fulfillment_status
orders_df_complete['fully_paid'] = fully_paid
orders_df_complete['original_total_price_set'] = original_total_price_set
orders_df_complete['payment_gateway_names'] = payment_gateway_names
orders_df_complete['refunds'] = refunds   
orders_df_complete['registered_source_url'] = registered_source_url
orders_df_complete['return_status'] = return_status
orders_df_complete['source_name'] = source_name
orders_df_complete['subtotal_line_items_quantity'] = subtotal_line_items_quantity
orders_df_complete['subtotal_price_set'] = subtotal_price_set   
orders_df_complete['total_discounts_set'] = total_discounts_set
orders_df_complete['total_price_set'] = total_price_set  
orders_df_complete['transactions'] = transactions  



In [16]:
# orders_df_complete

In [107]:
# result.content

# Eventual dataframe
test_bulk_data = []

# Read jsonl object
with open(contents) as f:
    for line in f:
        test_bulk_data.append(json.loads(line))

# Create dataframe from list
bulk_df_test = pd.DataFrame(test_bulk_data)

OSError: [Errno 63] File name too long: b'{"id":"gid:\\/\\/shopify\\/Order\\/11978147823984","name":"#555375","email":"genevieve.ziogas@gmail.com","createdAt":"2026-04-01T04:08:31Z","cancellation":null,"cancelledAt":null,"cancelReason":null,"cartDiscountAmountSet":null,"channelInformation":{"displayName":"Online Store"},"closed":false,"closedAt":null,"currentCartDiscountAmountSet":{"shopMoney":{"amount":"0.0"}},"currentShippingPriceSet":{"shopMoney":{"amount":"12.95"}},"currentSubtotalLineItemsQuantity":4,"currentSubtotalPriceSet":{"shopMoney":{"amount":"158.0"}},"currentTaxLines":[{"priceSet":{"shopMoney":{"amount":"11.11"}}},{"priceSet":{"shopMoney":{"amount":"4.1"}}}],"currentTotalAdditionalFeesSet":null,"currentTotalDiscountsSet":{"shopMoney":{"amount":"0.0"}},"currentTotalPriceSet":{"shopMoney":{"amount":"186.16"}},"currentTotalTaxSet":{"shopMoney":{"amount":"15.21"}},"discountCode":null,"discountCodes":[],"displayFinancialStatus":"PAID","displayFulfillmentStatus":"UNFULFILLED","f

In [5]:
new_df = utils.modify_columns(bulk_df)

#### Write JSONL to DataFrame

This returns a JSONL object.

Using [this thread](https://stackoverflow.com/questions/12451431/loading-and-parsing-a-json-file-with-multiple-json-objects) for code to read object.

Need to do this programmatically.

#### Writing results to BigQuery

In [38]:

schema = [
    bigquery.SchemaField("order_id", "STRING"),
    bigquery.SchemaField("order_name", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("created_at", "STRING"),
    bigquery.SchemaField("cancellation", "STRING"),
    bigquery.SchemaField("cancelled_at", "STRING"),
    bigquery.SchemaField("cancel_reason", "STRING"),
    bigquery.SchemaField("cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("channel_information", "STRING"),
    bigquery.SchemaField("closed", "BOOLEAN"),
    bigquery.SchemaField("closed_at", "STRING"),
    bigquery.SchemaField("current_cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("current_shipping_price_set", "NUMERIC"),
    bigquery.SchemaField("current_subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("current_subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("current_tax_lines", "STRING"),
    bigquery.SchemaField("current_total_additional_fees_set", "STRING"),
    bigquery.SchemaField("current_total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("current_total_price_set", "NUMERIC"),
    bigquery.SchemaField("current_total_tax_set", "NUMERIC"),
    bigquery.SchemaField("discount_code", "STRING"),
    bigquery.SchemaField("discount_codes", "STRING"),
    bigquery.SchemaField("display_financial_status", "STRING"),
    bigquery.SchemaField("display_fulfillment_status", "STRING"),
    bigquery.SchemaField("fully_paid", "BOOLEAN"),
    bigquery.SchemaField("original_total_price_set", "NUMERIC"),
    bigquery.SchemaField("payment_gateway_names", "STRING"),
    bigquery.SchemaField("refunds", "NUMERIC"),
    bigquery.SchemaField("registered_source_url", "STRING"),
    bigquery.SchemaField("return_status", "STRING"),
    bigquery.SchemaField("source_name", "STRING"),
    bigquery.SchemaField("subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("total_price_set", "NUMERIC"),
    bigquery.SchemaField("transactions", "STRING"),

    bigquery.SchemaField("closed_at_utc", "DATETIME"),
    bigquery.SchemaField("created_at_utc", "DATETIME"),
    bigquery.SchemaField("cancelled_at_utc", "DATETIME"),
]
table_id_raw_bulk = "glowing-market-295819.shopify.test_table_orders_raw_bulk_append"

job_config = bigquery.LoadJobConfig(
schema = [
    bigquery.SchemaField("id", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("name", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("email", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("createdAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancellation", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancelledAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancelReason", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cartDiscountAmountSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("channelInformation", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closed", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("closedAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentCartDiscountAmountSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentShippingPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentSubtotalLineItemsQuantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("currentSubtotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTaxLines", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentTotalAdditionalFeesSet", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentTotalDiscountsSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTotalTaxSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("discountCode", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("discountCodes", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("displayFinancialStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("displayFulfillmentStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("fullyPaid", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("originalTotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("paymentGatewayNames", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("refunds", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("registeredSourceUrl", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("returnStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("sourceName", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("subtotalLineItemsQuantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("subtotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("totalDiscountsSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("totalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("transactions", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closedAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("createdAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("cancelledAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
],
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(bulk_df,table_id_raw_bulk,job_config= job_config)
job.result()

table = client.get_table(table_id_raw_bulk)
print(
    "Loaded {} rows and {} columns to {}".format(
        table.num_rows, len(table.schema), table_id_raw_bulk
    )
)


/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 6337 rows and 39 columns to glowing-market-295819.shopify.test_table_orders_raw_bulk_append


### Bulk Operation - Yesterday

In [ ]:
import time

if (bulk_status_response['data']['currentBulkOperation']['status'] == 'COMPLETED' and bulk_status_response['data']['currentBulkOperation']['errorCode'] is None):
    print(bulk_status_response['data']['currentBulkOperation']['url'])

    url = bulk_status_response['data']['currentBulkOperation']['url']

r = requests.post(
    f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
    headers={
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
    },
    json={"query": q_bulk_status}
)

bulk_status_response = r.json()

##########



https://storage.googleapis.com/shopify-tiers-assets-prod-us-east1/bulk-operation-outputs/b0c1tkj7r47aqpmg1n5dl2njq75h-final?GoogleAccessId=assets-us-prod%40shopify-tiers.iam.gserviceaccount.com&Expires=1775737461&Signature=olunr8vMznx7Poune3dmteLQ9ucFg5o3Pp9lQ4Xt5B10u1ToW%2FmycN9jSwmpqBKNg4dfWkLpBbtMiykoYdgT3JpvYJMNXKG6L4um5MpBsesqBXCWcqqVZvokjsVLV6Nw0rkddq6NamYT3%2Bul3jShbX%2BKKwsW8By66Llzy3rkjlJ0AXujv4tvaN7L7nWrY%2BNYBTvEOC0qsg8Lcuzr0%2B8VrcnbKV0kuDFNHuJR5YUyuwtxMT%2BRwH3QHaolEx%2Fn6z4hSaeJVPZGBejMsjsY2A%2BnbAHv5SDEpnsOsclGFj0t%2Fj4%2BG4wIfrX3jzOYHzP9FeN2lP24If8X89IwnoDfqRe60g%3D%3D&response-content-disposition=attachment%3B+filename%3D%22bulk-7515585085808.jsonl%22%3B+filename%2A%3DUTF-8%27%27bulk-7515585085808.jsonl&response-content-type=application%2Fjsonl


In [92]:
def poll_for_result(bulk_status_query, interval_seconds=60, max_attempts=10):
    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}...")
        
        br = requests.post(
            f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
                headers={
                    "Content-Type": "application/json",
                    "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
                },
                json={"query": bulk_status_query}
            )

        bulk_status_response = br.json()
        
        if (bulk_status_response['data']['currentBulkOperation']['status'] == 'COMPLETED' and bulk_status_response['data']['currentBulkOperation']['errorCode'] is None):
            print("Done! File available at:", bulk_status_response['data']['currentBulkOperation']['url'])
            return bulk_status_response['data']['currentBulkOperation']['url']
            
        print(f"Status: {bulk_status_response['data']['currentBulkOperation']['status']}. Retrying in {interval_seconds}s...")
        time.sleep(interval_seconds)

        raise TimeoutError("Job did not complete within the allowed attempts.")

In [94]:
url_results = poll_for_result(q_bulk_status)

Attempt 1/10...
Done! File available at: https://storage.googleapis.com/shopify-tiers-assets-prod-us-east1/bulk-operation-outputs/b0c1tkj7r47aqpmg1n5dl2njq75h-final?GoogleAccessId=assets-us-prod%40shopify-tiers.iam.gserviceaccount.com&Expires=1775737461&Signature=olunr8vMznx7Poune3dmteLQ9ucFg5o3Pp9lQ4Xt5B10u1ToW%2FmycN9jSwmpqBKNg4dfWkLpBbtMiykoYdgT3JpvYJMNXKG6L4um5MpBsesqBXCWcqqVZvokjsVLV6Nw0rkddq6NamYT3%2Bul3jShbX%2BKKwsW8By66Llzy3rkjlJ0AXujv4tvaN7L7nWrY%2BNYBTvEOC0qsg8Lcuzr0%2B8VrcnbKV0kuDFNHuJR5YUyuwtxMT%2BRwH3QHaolEx%2Fn6z4hSaeJVPZGBejMsjsY2A%2BnbAHv5SDEpnsOsclGFj0t%2Fj4%2BG4wIfrX3jzOYHzP9FeN2lP24If8X89IwnoDfqRe60g%3D%3D&response-content-disposition=attachment%3B+filename%3D%22bulk-7515585085808.jsonl%22%3B+filename%2A%3DUTF-8%27%27bulk-7515585085808.jsonl&response-content-type=application%2Fjsonl


#### Writing JSONL data to DataFrame

In [9]:
bulk_df.shape

(194, 36)

#### Modifying Columns Before Loading to BigQuery 

In [73]:
# Column Definitions
b_num_columns = ['cartDiscountAmountSet','currentCartDiscountAmountSet','currentShippingPriceSet','currentSubtotalPriceSet','currentTotalDiscountsSet','currentTotalPriceSet','currentTotalTaxSet','originalTotalPriceSet','subtotalPriceSet','totalDiscountsSet','totalPriceSet']

b_date_cols = ['closedAt','createdAt','cancelledAt']

b_dict_columns = ["cancellation", "channelInformation","currentTaxLines","discountCodes","paymentGatewayNames","refunds","transactions"] 


# For dict/list items
for col in b_dict_columns:

    if col == "refunds":
        bulk_df[col] = bulk_df[col].apply(lambda x: sum(float(d['totalRefundedSet']['shopMoney']['amount']) for d in x) if x else 0)   
    elif col == "currentTaxLines":
        bulk_df[col] = bulk_df[col].apply(lambda x: json.dumps([float(d['priceSet']['shopMoney']['amount']) for d in x]) if x else None)
    # elif col == "discount_codes":
    #     bulk_df[col] = bulk_df[col].apply(lambda x: [d for d in x] if x else None)
    else:    
        bulk_df[col] = bulk_df[col].apply(lambda x: json.dumps(x) if (isinstance(x, dict) or isinstance(x, list)) else None)


# Columns that need to be numbers
# Columns with strings that need to be converted to float

for col in b_num_columns:
    if col == "cartDiscountAmountSet":
        bulk_df[col] = bulk_df[col].apply(lambda x: float(x['shopMoney']['amount']) if x else 0)    
    else:
        bulk_df[col] = bulk_df[col].apply(lambda x: float(x['shopMoney']['amount']))

# Date Columns
for col in b_date_cols:
    col_utc = col + '_utc'

    if col == 'cancelledAt' or col == 'closedAt':
        bulk_df[col_utc] = bulk_df[col].apply(lambda x: pd.to_datetime(x) if x is not None else None)
    else:
        bulk_df[col_utc] = bulk_df[col].apply(lambda x: pd.to_datetime(x))

In [17]:
# bulk_df[['cartDiscountAmountSet','currentCartDiscountAmountSet','currentShippingPriceSet','currentSubtotalPriceSet','currentTotalDiscountsSet','currentTotalPriceSet','currentTotalTaxSet','originalTotalPriceSet','subtotalPriceSet','totalDiscountsSet','totalPriceSet']]

In [18]:
# bulk_df[['closedAt','createdAt','cancelledAt']]

In [19]:
# bulk_df[["cancellation", "channelInformation","currentTaxLines","discountCodes","paymentGatewayNames","refunds","transactions"] ]

In [14]:
bulk_df.dtypes

id                                  object
name                                object
email                               object
createdAt                           object
cancellation                        object
cancelledAt                         object
cancelReason                        object
cartDiscountAmountSet               object
channelInformation                  object
closed                                bool
closedAt                            object
currentCartDiscountAmountSet        object
currentShippingPriceSet             object
currentSubtotalLineItemsQuantity     int64
currentSubtotalPriceSet             object
currentTaxLines                     object
currentTotalAdditionalFeesSet       object
currentTotalDiscountsSet            object
currentTotalPriceSet                object
currentTotalTaxSet                  object
discountCode                        object
discountCodes                       object
displayFinancialStatus              object
displayFulf

In [6]:
def modify_columns(df):

    # Column Definitions
    b_num_columns = ['cartDiscountAmountSet','currentCartDiscountAmountSet','currentShippingPriceSet','currentSubtotalPriceSet','currentTotalDiscountsSet','currentTotalPriceSet','currentTotalTaxSet','originalTotalPriceSet','subtotalPriceSet','totalDiscountsSet','totalPriceSet']
    b_date_cols = ['closedAt','createdAt','cancelledAt']
    b_dict_columns = ["cancellation", "channelInformation","currentTaxLines","discountCodes","paymentGatewayNames","refunds","transactions"] 

    # For dict/list items
    for col in b_dict_columns:

        if col == "refunds":
            df[col] = df[col].apply(lambda x: sum(float(d['totalRefundedSet']['shopMoney']['amount']) for d in x) if x else 0)   
        elif col == "currentTaxLines":
            df[col] = df[col].apply(lambda x: json.dumps([float(d['priceSet']['shopMoney']['amount']) for d in x]) if x else None)
        else:    
            df[col] = df[col].apply(lambda x: json.dumps(x) if (isinstance(x, dict) or isinstance(x, list)) else None)


    # Columns that need to be numbers
    # Columns with strings that need to be converted to float

    for col in b_num_columns:
        if col == "cartDiscountAmountSet":
            df[col] = df[col].apply(lambda x: float(x['shopMoney']['amount']) if x else 0)    
        else:
            df[col] = df[col].apply(lambda x: float(x['shopMoney']['amount']))

    # Date Columns
    for col in b_date_cols:
        col_utc = col + '_utc'

        if col == 'cancelledAt' or col == 'closedAt':
            df[col_utc] = df[col].apply(lambda x: pd.to_datetime(x) if x is not None else None)
        else:
            df[col_utc] = df[col].apply(lambda x: pd.to_datetime(x))

In [20]:
# bulk_df[['cartDiscountAmountSet','currentCartDiscountAmountSet','currentShippingPriceSet','currentSubtotalPriceSet','currentTotalDiscountsSet','currentTotalPriceSet','currentTotalTaxSet','originalTotalPriceSet','subtotalPriceSet','totalDiscountsSet','totalPriceSet']]

In [21]:
# bulk_df[['closedAt_utc','createdAt_utc','cancelledAt_utc']]

In [22]:
# bulk_df[["cancellation", "channelInformation","currentTaxLines","discountCodes","paymentGatewayNames","refunds","transactions"]]

#### Loading to BigQuery

Difference is that write_disposition will append to existing dataframe. That way I can reutilize this code.

In [76]:
schema = [
    bigquery.SchemaField("order_id", "STRING"),
    bigquery.SchemaField("order_name", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("created_at", "STRING"),
    bigquery.SchemaField("cancellation", "STRING"),
    bigquery.SchemaField("cancelled_at", "STRING"),
    bigquery.SchemaField("cancel_reason", "STRING"),
    bigquery.SchemaField("cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("channel_information", "STRING"),
    bigquery.SchemaField("closed", "BOOLEAN"),
    bigquery.SchemaField("closed_at", "STRING"),
    bigquery.SchemaField("current_cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("current_shipping_price_set", "NUMERIC"),
    bigquery.SchemaField("current_subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("current_subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("current_tax_lines", "STRING"),
    bigquery.SchemaField("current_total_additional_fees_set", "STRING"),
    bigquery.SchemaField("current_total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("current_total_price_set", "NUMERIC"),
    bigquery.SchemaField("current_total_tax_set", "NUMERIC"),
    bigquery.SchemaField("discount_code", "STRING"),
    bigquery.SchemaField("discount_codes", "STRING"),
    bigquery.SchemaField("display_financial_status", "STRING"),
    bigquery.SchemaField("display_fulfillment_status", "STRING"),
    bigquery.SchemaField("fully_paid", "BOOLEAN"),
    bigquery.SchemaField("original_total_price_set", "NUMERIC"),
    bigquery.SchemaField("payment_gateway_names", "STRING"),
    bigquery.SchemaField("refunds", "NUMERIC"),
    bigquery.SchemaField("registered_source_url", "STRING"),
    bigquery.SchemaField("return_status", "STRING"),
    bigquery.SchemaField("source_name", "STRING"),
    bigquery.SchemaField("subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("total_price_set", "NUMERIC"),
    bigquery.SchemaField("transactions", "STRING"),

    bigquery.SchemaField("closed_at_utc", "DATETIME"),
    bigquery.SchemaField("created_at_utc", "DATETIME"),
    bigquery.SchemaField("cancelled_at_utc", "DATETIME"),
]
table_id_raw_bulk = "glowing-market-295819.shopify.test_table_orders_raw_bulk_append"

job_config = bigquery.LoadJobConfig(
schema = [
    bigquery.SchemaField("id", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("name", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("email", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("createdAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancellation", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancelledAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancelReason", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cartDiscountAmountSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("channelInformation", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closed", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("closedAt", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentCartDiscountAmountSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentShippingPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentSubtotalLineItemsQuantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("currentSubtotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTaxLines", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentTotalAdditionalFeesSet", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("currentTotalDiscountsSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("currentTotalTaxSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("discountCode", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("discountCodes", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("displayFinancialStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("displayFulfillmentStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("fullyPaid", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("originalTotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("paymentGatewayNames", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("refunds", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("registeredSourceUrl", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("returnStatus", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("sourceName", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("subtotalLineItemsQuantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("subtotalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("totalDiscountsSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("totalPriceSet", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("transactions", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closedAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("createdAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("cancelledAt_utc", bigquery.enums.SqlTypeNames.DATETIME),
],
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(bulk_df,table_id_raw_bulk,job_config= job_config)
job.result()

table = client.get_table(table_id_raw_bulk)
print(
    "Loaded {} rows and {} columns to {}".format(
        table.num_rows, len(table.schema), table_id_raw_bulk
    )
)


/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 6531 rows and 39 columns to glowing-market-295819.shopify.test_table_orders_raw_bulk_append
